# ⚡ V10 Scraper (Tuned): High Yield & Quality

**Goal:** Balance V8's high volume with V10's speed.
**Updates:** Relaxed filters to catch shorter articles (250+ chars) and deep links.

### 🛠️ Architecture (Strict Order)
1.  **Discovery Phase (The Scout):** 
    *   *Step 1:* Check RSS Feed (Fastest, Cleanest)
    *   *Step 2:* Check Sitemap (Backup)
    *   *Step 3:* Scan HTML (Broader scan for any link > 15 chars)
2.  **Scraping Phase (The Worker):**
    *   *Tier 1:* Async HTTP (Trafilatura) -> Fast
    *   *Tier 2:* Headless Browser (Playwright) -> Powerful (only if Tier 1 fails)
3.  **Cleaning Phase (The Janitor):**
    *   Apply V8 Regex rules to strip "Subscribe", "Share", and Ads.

---

In [1]:
# CELL 1: SETUP & IMPORTS
import asyncio
import aiohttp
import logging
import sys
import os
import psutil
import json
import hashlib
import re
import pandas as pd
from datetime import datetime
from typing import List, Dict, Optional, Any
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup

# Third-party libraries
try:
    import trafilatura
    from trafilatura import feeds
    import feedparser
    from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
    import nest_asyncio
except ImportError as e:
    print(f"❌ Missing critical library: {e}")
    print("Run: pip install aiohttp trafilatura playwright psutil pandas nest_asyncio feedparser beautifulsoup4")

# KEY FIX: Apply nest_asyncio to prevent Jupyter from crashing with Playwright
nest_asyncio.apply()

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    handlers=[logging.FileHandler("scraper_service.log", encoding='utf-8'), logging.StreamHandler()]
)
logger = logging.getLogger("V10_Rebuilt")

print("✅ Setup Complete & Async Loop Patched.")

✅ Setup Complete & Async Loop Patched.


In [2]:
# CELL 2: CONFIGURATION & UTILITIES (V8 PORT)

class SystemConfig:
    def __init__(self):
        # Resource Limits
        self.MAX_BROWSERS = 2  # Conservative limit to prevent crashes
        self.MAX_ASYNC_WORKERS = 25 # Reduced from 40 to avoid WAF blocking
        
        # Paths
        # Uses V8's source list (Proven Quality)
        self.SOURCES_PATH = "../v8_Scraper_Parallel/sources.json"
        # NEW: Using v12 to ensure fresh start
        self.CSV_OUTPUT_PATH = "scraped_news_v12.csv"
        self.USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        
        # V8 Keywords
        self.KEYWORDS = ["A.I.", "Google", "Microsoft", "Nvidia", "Energy", "Chipset"]
        
        # V8 Cleaning Patterns (The "Janitor")
        self.PATTERNS_TO_REMOVE = [
            r'Share this article.*',
            r'Subscribe to.*newsletter',
            r'Sign up for.*',
            r'Follow us on.*',
            r'Read more:.*',
            r'Related articles.*',
            r'Advertisement',
            r'ADVERTISEMENT',
            r'Click here to.*',
            r'© 20\d{2}.*'
        ]

CONFIG = SystemConfig()

# --- V8 TEXT CLEANING UTILITIES ---
def strip_html(text: str) -> str:
    """Strip HTML tags using BeautifulSoup (Better than Regex)."""
    if not text: return ""
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')

def clean_text(text: str) -> str:
    """Apply V8's rigorous cleaning rules."""
    if not text: return ""
    # 1. Strip HTML
    text = strip_html(text)
    # 2. Normalize Whitespace
    text = re.sub(r'\s+', ' ', text)
    # 3. Remove Boilerplate
    for pattern in CONFIG.PATTERNS_TO_REMOVE:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()

print("✅ Configuration & V8 Cleaning Logic Loaded.")

✅ Configuration & V8 Cleaning Logic Loaded.


In [3]:
# CELL 3: DISCOVERY SERVICE (STRICT WATERFALL)

class DiscoveryService:
    """
    The 'Scout'. Strictly follows V8's Waterfall Precedence:
    1. RSS Feed (Best)
    2. Sitemap (Good)
    3. HTML Link Scan (Last Resort)
    """
    
    async def discover_rss(self, source_url: str) -> Optional[str]:
        """Find RSS feed URL."""
        # 1. Check Trafilatura's auto-discovery
        try:
             possible = feeds.find_feed_urls(source_url)
             if possible:
                 # Returns list in newer versions
                 return possible[0] if isinstance(possible, list) else possible
        except: pass
        
        # 2. Check common paths manually
        common_paths = ["/rss", "/feed", "/feeds", "/rss.xml"]
        async with aiohttp.ClientSession() as session:
            for path in common_paths:
                target = urljoin(source_url, path)
                try:
                    async with session.get(target, timeout=5) as resp:
                        if resp.status == 200 and 'xml' in resp.headers.get('Content-Type', ''):
                            return target
                except: pass
        return None

    async def parse_rss(self, feed_url: str) -> List[Dict]:
        """Parse the RSS feed for articles."""
        articles = []
        try:
            # Feedparser is synchronous, but fast. We wrap it or just run it.
            # For simplicity in this notebook, we run it directly (blocking for ms is negligible here)
            d = feedparser.parse(feed_url)
            for entry in d.entries[:10]: # Top 10 freshest
                articles.append({
                    "url": entry.link,
                    "title": entry.get('title', 'Unknown'),
                    "date": entry.get('published', datetime.now().isoformat()),
                    "method": "rss"
                })
        except Exception as e:
            logger.warning(f"RSS Parse Failed: {e}")
        return articles

    async def scan_html_fallback(self, source_url: str) -> List[Dict]:
        """Scrape the homepage and look for article links (V8 Fallback)."""
        found = []
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(source_url, headers={'User-Agent': CONFIG.USER_AGENT}, timeout=10) as resp:
                    if resp.status != 200: return []
                    html = await resp.text()
            
            soup = BeautifulSoup(html, 'html.parser')
            base_domain = urlparse(source_url).netloc
            
            for a in soup.find_all('a', href=True):
                href = a['href']
                full_url = urljoin(source_url, href)
                parsed = urlparse(full_url)
                
                # V8 HEURISTICS (RELAXED): 
                # Just check if it looks like a deep link (> 15 chars)
                if parsed.netloc == base_domain and len(parsed.path) > 15:
                    found.append({
                        "url": full_url,
                        "title": a.get_text(strip=True),
                        "date": datetime.now().isoformat(),
                        "method": "html_scan"
                    })
        except Exception as e:
            logger.warning(f"HTML Scan Failed: {e}")
            
        # Deduplicate
        unique = list({v['url']:v for v in found}.values())
        return unique[:15]

    async def find_articles(self, source_url: str) -> List[Dict]:
        """Orchestrate the Waterfall."""
        # STEP 1: RSS
        rss_url = await self.discover_rss(source_url)
        if rss_url:
            logger.info(f"✅ Found RSS: {rss_url}")
            return await self.parse_rss(rss_url)
        
        # STEP 2: HTML Fallback (if RSS missing)
        logger.info(f"⚠️ No RSS. Falling back to HTML Scan for {source_url}")
        return await self.scan_html_fallback(source_url)

DISCOVERY = DiscoveryService()
print("✅ Discovery Service (Waterfall) Initialized.")

✅ Discovery Service (Waterfall) Initialized.


In [4]:
# CELL 4: THE HYBRID ENGINE (ASYNC + PLAYWRIGHT)

class HybridScraper:
    def __init__(self):
        self.browser_sem = asyncio.Semaphore(CONFIG.MAX_BROWSERS)
        self.async_sem = asyncio.Semaphore(CONFIG.MAX_ASYNC_WORKERS)
        
        self.columns = [
            "source", "headline", "author", "url", "published", 
            "matched_keywords", "content_snippet", "full_content", "method", "scraped_at"
        ]
        
        # Init CSV Output
        if not os.path.exists(CONFIG.CSV_OUTPUT_PATH):
            pd.DataFrame(columns=self.columns).to_csv(CONFIG.CSV_OUTPUT_PATH, index=False)

    def get_keywords(self, text: str) -> List[str]:
        return [k for k in CONFIG.KEYWORDS if k.lower() in text.lower()]

    def save_result(self, data: Dict):
        # CLEANING HAPPENS HERE (The Janitor)
        final_content = clean_text(data['content'])
        
        row = {
            "source": urlparse(data['url']).netloc,
            "headline": data['title'],
            "author": data.get('author'),
            "url": data['url'],
            "published": data['date'],
            "matched_keywords": ", ".join(data['keywords']),
            "content_snippet": final_content[:300].replace("\n", " "),
            "full_content": final_content,
            "method": data['method'],
            "scraped_at": datetime.now().isoformat()
        }
        # Enforce Column Order
        pd.DataFrame([row], columns=self.columns).to_csv(CONFIG.CSV_OUTPUT_PATH, mode='a', header=False, index=False)
        logger.info(f"💾 Saved: {data['title'][:40]}...")

    async def scrape_article(self, item: Dict):
        """Tier 1 (Trafilatura) -> Tier 2 (Playwright)"""
        url = item['url']
        
        # TIER 1: FAST FETCH (Trafilatura)
        async with self.async_sem:
            try:
                async with aiohttp.ClientSession() as session:
                    async with session.get(url, headers={'User-Agent': CONFIG.USER_AGENT}, timeout=10) as resp:
                        if resp.status == 200:
                            html = await resp.text()
                            extracted = trafilatura.extract(html)
                            # RELAXED LIMIT: 250 characters (was 500)
                            if extracted and len(extracted) > 250:
                                kws = self.get_keywords(extracted)
                                if kws:
                                    item['content'] = extracted
                                    item['keywords'] = kws
                                    item['method'] += " + trafilatura"
                                    self.save_result(item)
                                    return
            except Exception: pass

        # TIER 2: PLAYWRIGHT (If Tier 1 failed or no keywords)
        async with self.browser_sem:
            logger.info(f"🎭 Triggering Playwright for {url}")
            try:
                async with async_playwright() as p:
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    await page.goto(url, wait_until="domcontentloaded", timeout=20000)
                    content = await page.content()
                    extracted = trafilatura.extract(content)
                    await browser.close()
                    
                    # RELAXED LIMIT
                    if extracted and len(extracted) > 250:
                         kws = self.get_keywords(extracted)
                         if kws:
                             item['content'] = extracted
                             item['keywords'] = kws
                             item['method'] += " + playwright"
                             self.save_result(item)
            except Exception as e:
                logger.error(f"Playwright Failed {url}: {e}")

    async def process_source(self, source_url: str):
        """Full Pipeline: Discover -> Scrape"""
        articles = await DISCOVERY.find_articles(source_url)
        if not articles:
            logger.info(f"❌ Failed to find any articles for {source_url}")
            return
            
        logger.info(f"🔍 Processing {len(articles)} articles for {source_url}...")
        tasks = [self.scrape_article(a) for a in articles]
        await asyncio.gather(*tasks)

SCRAPER = HybridScraper()
print("✅ Hybrid Engine Ready.")

✅ Hybrid Engine Ready.


In [5]:
# CELL 5: MAIN EXECUTION

async def main():
    # Load Sources
    with open(CONFIG.SOURCES_PATH, 'r') as f:
        sources = json.load(f)['sources']
    
    print(f"🚀 Starting V10 (Rebuilt) on {len(sources)} sources...")
    
    # Batching to avoid event loop overload
    BATCH_SIZE = 5
    for i in range(0, len(sources), BATCH_SIZE):
        batch = sources[i:i+BATCH_SIZE]
        print(f"\n--- Batch {i//BATCH_SIZE + 1} ({len(batch)} sources) ---")
        tasks = [SCRAPER.process_source(s['url']) for s in batch]
        await asyncio.gather(*tasks)
    
    print(f"\n🏁 Scraping Complete. Check '{CONFIG.CSV_OUTPUT_PATH}'.")

# await main()
print("▶️  Run 'await main()' to start.")

▶️  Run 'await main()' to start.


In [6]:
# CELL 6: ANALYTICS & REPORTING
def show_analytics():
    if not os.path.exists(CONFIG.CSV_OUTPUT_PATH):
        print(f"⚠️ No CSV found at {CONFIG.CSV_OUTPUT_PATH}. Run main() first.")
        return
        
    df = pd.read_csv(CONFIG.CSV_OUTPUT_PATH)
    print(f"📊 Total Articles: {len(df)}")
    print("\n📈 Method Breakdown:")
    print(df['method'].value_counts())
    print("\n🏆 Top Sources:")
    print(df['source'].value_counts().head(5))
    
    if not df.empty:
        print("\n🧐 Quality Check (First Snippet):")
        # Use safe access
        if 'content_snippet' in df.columns:
            print(df.iloc[0]['content_snippet'])
        else:
            print("⚠️ 'content_snippet' column missing (Schema error?)")

# show_analytics()
print("▶️  Run 'show_analytics()' to view report.")

▶️  Run 'show_analytics()' to view report.
